# Cryptographic Hash

### Lucie Spanbauer, Ashraf Mirza, Monish Sreekanth, Kmm9660

# Part I — Merkle Trees and Data Integrity

A leaf is the SHA-256 digest of a file's bytes. Each parent is SHA-256 of the concatenation of its two child digests. If a level has an odd number of hashes, its last hash is duplicated. Hexadecimal digests are concatenated as bytes, not as their printable hexadecimal characters.


In [ ]:
# Part I, Experiment 1 — Merkle tree with four leaf files
import hashlib

def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def leaf_hash(content: str) -> bytes:
    return sha256(content.encode("utf-8"))

def merkle_levels(file_contents):
    # Each level is stored left-to-right; level 0 contains the file/leaf hashes.
    levels = [[leaf_hash(content) for content in file_contents]]
    while len(levels[-1]) > 1:
        current = levels[-1]
        parents = []
        for i in range(0, len(current), 2):
            left = current[i]
            right = current[i + 1] if i + 1 < len(current) else current[i]
            parents.append(sha256(left + right))
        levels.append(parents)
    return levels

def display_tree(file_names, levels):
    print("Leaves (file content hashes):")
    for name, digest in zip(file_names, levels[0]):
        print(f"  {name}: {digest.hex()}")
    for level_index in range(1, len(levels)):
        print(f"Level {level_index} ({len(levels[level_index])} node(s)):")
        for node_index, digest in enumerate(levels[level_index]):
            print(f"  node {node_index}: {digest.hex()}")
    print("Merkle root:", levels[-1][0].hex())

files4 = {
    "file1.txt": "Alice sends 10 tokens",
    "file2.txt": "Bob sends 20 tokens",
    "file3.txt": "Carol sends 30 tokens",
    "file4.txt": "Dave sends 40 tokens",
}
names4 = list(files4)
levels4 = merkle_levels(list(files4.values()))
display_tree(names4, levels4)


Leaves (file content hashes):
  file1.txt: 127b5fed9d96a97f31f09212b0568e0498c84c76b449f2f278d1c07e3c58d80d
  file2.txt: ec9a71872ee053aa82972bbce50f4a51d35c2bd2b5a90c7b5c35ae4eba8ad198
  file3.txt: 31ba9ddb0dfcf1902d5121a605a8119dd5fe41df850e8aa4ec5a2c8ee63f7714
  file4.txt: feac8a16a4560a3dd635ff3ee4f76061d421b3085295f7423db2f04f1460bb20
Level 1 (2 node(s)):
  node 0: 84a0f2858188b2a1c18a9549f908f8025c2fea8da559633f9843880513cbee13
  node 1: cfaf782eccc3c382a02d38d7098809d064b2c6a5e321b1ed7feb388a3820c8ff
Level 2 (1 node(s)):
  node 0: d6039618a6cd11680adf359d21f81611a73103e502ac0e64c1d9bf08113fe5ec
Merkle root: d6039618a6cd11680adf359d21f81611a73103e502ac0e64c1d9bf08113fe5ec


In [ ]:
# Part I, Experiment 2 — Merkle tree with six leaf files (odd node at an intermediate level)
files6 = {
    "file1.txt": "record 1: alpha",
    "file2.txt": "record 2: bravo",
    "file3.txt": "record 3: charlie",
    "file4.txt": "record 4: delta",
    "file5.txt": "record 5: echo",
    "file6.txt": "record 6: foxtrot",
}
names6 = list(files6)
levels6 = merkle_levels(list(files6.values()))
display_tree(names6, levels6)
print("Note: with six leaves, the first parent level has three nodes; the last of those three is duplicated when building the root.")


Leaves (file content hashes):
  file1.txt: 77d67fd296f4a8f9cc8a02b344bfebdd08a870de3295f24c4a1b95a367b4b1cb
  file2.txt: 248274744cb0fcf226d3021aebebf5aa2bb2cf4ea5cbcb43cdc84911a8885557
  file3.txt: 8bbe6e3cdf88d62677b18392bfeea6d0d4d256226c931b04bfb8eb26a0387231
  file4.txt: 9dfdb37014b9ca55b1d00bf2096d343667c568194693b0ee188fe4112cea8863
  file5.txt: b2f7a9fe70244c9aea601faf11617256cf0a62b877893453ed8e6f8960fe7cd7
  file6.txt: a20e4cd7623536b8c2f73fded95f3e5939b7362cd5f060d152a3da7941668145
Level 1 (3 node(s)):
  node 0: b700e3dd285f0928993188e71c7a9729fa58c9ab0cee06fce561e3fcbb04c6c0
  node 1: 2615a943e2226bb967632aec60ec0ca213d8ad92cf782c3c37ad8376ff3f1806
  node 2: 02e9a11547d7d6dc028f4e9919085317e5f64bf4dd6a33885d584e52e2b9962b
Level 2 (2 node(s)):
  node 0: 147c89e5cb42a4f2e8e3ec82f08afa2b4a26a1c66cffbd01c46de4ca2d29ac4d
  node 1: 69eab188827fc09f7c17d799422dd2b05ec9adbe53f7df8bf8d55f6f47c4f324
Level 3 (1 node(s)):
  node 0: a800e833581a7ca0fe8f8cb3c0a74737b5144815a6fddc62647b2c

In [ ]:
# Part I, Experiment 3 — Modify one file and compare roots
modified_files4 = dict(files4)
modified_files4["file2.txt"] = "Bob sends 200 tokens"  # one content change
modified_levels4 = merkle_levels(list(modified_files4.values()))

original_root = levels4[-1][0].hex()
modified_root = modified_levels4[-1][0].hex()
print("Original root: ", original_root)
print("Modified root:", modified_root)
print("Roots equal?  ", original_root == modified_root)
print("\nChanged nodes by level (compare original vs modified tree):")
for level_index, (old_level, new_level) in enumerate(zip(levels4, modified_levels4)):
    changed = [i for i, (old, new) in enumerate(zip(old_level, new_level)) if old != new]
    print(f"  Level {level_index}: changed node indices {changed}")


Original root:  d6039618a6cd11680adf359d21f81611a73103e502ac0e64c1d9bf08113fe5ec
Modified root: 7e3d08f20d4d4d769e0af2d0ce9dace6c5ee5230fc17620f75e1afdb5cfeaf69
Roots equal?   False

Changed nodes by level (compare original vs modified tree):
  Level 0: changed node indices [1]
  Level 1: changed node indices [0]
  Level 2: changed node indices [0]


# Part II — Hash Collisions and the Birthday Effect

For each input, this experiment computes SHA-256 and keeps only the first `k` bits. The implementation supports bit lengths that are not multiples of four by taking the required leading bits from the digest. Each trial generates distinct random text inputs and stops at the first repeated truncated hash. The reported attempt count includes the input that produced the collision.


In [ ]:
# Part II — Collision search implementation and repeated experiments
import random
import statistics
import hashlib

def truncated_sha256(text, k):
    digest_int = int.from_bytes(hashlib.sha256(text.encode("utf-8")).digest(), "big")
    return digest_int >> (256 - k)  # integer containing the leading k bits

def collision_trial(k, rng):
    seen = {}
    attempts = 0
    while True:
        attempts += 1
        # Counter guarantees each trial input is distinct; random text supplies varying content.
        message = f"trial={rng.getrandbits(64)}; attempt={attempts}; salt={rng.getrandbits(64)}"
        value = truncated_sha256(message, k)
        if value in seen:
            return attempts, seen[value], message, value
        seen[value] = message

rng = random.Random(20261029)
TRIALS_PER_K = 30
collision_results = {}
for k in (4, 8, 12):
    counts = []
    for _ in range(TRIALS_PER_K):
        attempts, first_message, second_message, digest_value = collision_trial(k, rng)
        counts.append(attempts)
    collision_results[k] = counts
    print(f"k={k} bits, trials={TRIALS_PER_K}")
    print(f"  minimum attempts: {min(counts)}")
    print(f"  maximum attempts: {max(counts)}")
    print(f"  average attempts: {statistics.mean(counts):.2f}")
    print(f"  all trial counts: {counts}")


k=4 bits, trials=30
  minimum attempts: 2
  maximum attempts: 15
  average attempts: 5.77
  all trial counts: [2, 5, 5, 6, 7, 4, 4, 8, 4, 6, 8, 6, 5, 3, 2, 8, 5, 8, 3, 7, 8, 3, 3, 6, 8, 7, 5, 2, 15, 10]
k=8 bits, trials=30
  minimum attempts: 4
  maximum attempts: 41
  average attempts: 21.27
  all trial counts: [16, 28, 33, 21, 30, 20, 8, 6, 11, 13, 28, 20, 10, 30, 4, 35, 18, 35, 22, 33, 28, 28, 41, 15, 8, 4, 31, 29, 22, 11]
k=12 bits, trials=30
  minimum attempts: 28
  maximum attempts: 196
  average attempts: 87.70
  all trial counts: [99, 48, 95, 99, 104, 130, 37, 50, 196, 83, 81, 103, 102, 114, 80, 88, 32, 112, 124, 102, 62, 46, 60, 51, 116, 67, 28, 133, 78, 111]


# Part III — Hash Puzzles and Proof-of-Work

The puzzle hashes the UTF-8 bytes of `message || nonce`, where the nonce is represented as decimal text. A valid answer has the required number of leading zero **bits**, not necessarily leading zero hexadecimal characters. The search starts at nonce zero and checks candidates sequentially.


In [ ]:
# Part III — Proof-of-work search for 1, 2, and 4 leading zero bits
def proof_of_work(message, required_zero_bits):
    nonce = 0
    prefix = message.encode("utf-8")
    while True:
        digest = hashlib.sha256(prefix + str(nonce).encode("ascii")).digest()
        digest_int = int.from_bytes(digest, "big")
        if digest_int >> (256 - required_zero_bits) == 0:
            return nonce, digest.hex(), nonce + 1
        nonce += 1

pow_message = "Cryptography project demo:"
for zero_bits in (1, 2, 4):
    nonce, digest_hex, attempts = proof_of_work(pow_message, zero_bits)
    print(f"Required leading zero bits: {zero_bits}")
    print(f"  nonce found: {nonce}")
    print(f"  attempts checked: {attempts}")
    print(f"  SHA-256(message || nonce): {digest_hex}")
    print(f"  verified prefix: {bin(int(digest_hex, 16))[2:].zfill(256)[:zero_bits]}")


Required leading zero bits: 1
  nonce found: 0
  attempts checked: 1
  SHA-256(message || nonce): 786d042123440b493ef7b5aa0877cc6f9183e28ce195f024107e45b46c78894e
  verified prefix: 0
Required leading zero bits: 2
  nonce found: 4
  attempts checked: 5
  SHA-256(message || nonce): 18c653fe932d595d439c96e882996b2ce5abf82e144f89d09d5a55e826e343d6
  verified prefix: 00
Required leading zero bits: 4
  nonce found: 36
  attempts checked: 37
  SHA-256(message || nonce): 0736a10d901198f590dfc395c9d249fddba18675c3f867fdf8c026fd1ae4e125
  verified prefix: 0000
